# Module 0 - Data acquisition and experimental record

**Goal.** Record the pupillary light reflex (PLR) on video and build the working database for the term.
This notebook does *not* acquire video (that is done with the phone); it **inventories and validates** the recordings so
problems (wrong duration, variable frame rate, missing records) are found before processing anything.

**Protocol reminder (course guide)**

| Record | Transition | What must happen |
|---|---|---|
| A (`A_n.mp4`) | dark -> light | start in low light, switch the light on / bring it close: fast constriction |
| B (`B_n.mp4`) | light -> dark | pupil already constricted, switch the light off / move it away: slow re-dilation |

* One eye only, filling most of the frame, head still, ~30 s per record, avoid blinking at the exact moment of the light change.
* 2-3 records of type A and 2-3 of type B.

All paths and thresholds come from `config.yaml`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():      # notebook launched from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from psib.config import load_config
from psib import video_io as vio
from psib.stimulus import detect_stimulus_onset
from psib.plotting import save_fig

cfg = load_config(ROOT / "config.yaml")
m0 = cfg["module0"]
print("Video folder:", cfg["video_dir"])

## 0.1 Inventory of the recordings

In [ ]:
paths = vio.list_videos(cfg["video_dir"], cfg["video_patterns"])
if not paths:
    raise FileNotFoundError(f"No videos matching {cfg['video_patterns']} in {cfg['video_dir']}")

# Decode every video once: timestamps + global brightness per frame
scans = {p.name: vio.scan_video(p, m0["brightness_downscale"]) for p in paths}

rows = []
for p in paths:
    s = scans[p.name]
    dt = np.diff(s["timestamps_s"])
    rows.append({
        "file": p.name,
        "type": vio.record_type(p),
        "width": s["width"], "height": s["height"],
        "frames": s["n_frames"],
        "duration_s": round(s["duration_s"], 2),
        "fps_nominal": round(s["fps_nominal"], 2),
        "fps_measured": round(len(dt) / (s["timestamps_s"][-1] - s["timestamps_s"][0]), 2),
        "max_gap_ms": round(1000 * dt.max()),
        "size_MB": round(p.stat().st_size / 1e6, 2),
    })
inventory = pd.DataFrame(rows).set_index("file")
inventory

## 0.2 Protocol checks

`fps_measured` is computed from the real frame timestamps. Phones often record at *variable* frame rate
(frames are dropped when the scene is dark or the CPU is busy), so a **time axis must always be built from
timestamps, never from `frame_index / fps`** (this matters for latency and velocity in Module 3).

In [ ]:
def flag(ok, msg_ok, msg_bad):
    print(("[ OK ]  " if ok else "[WARN]  ") + (msg_ok if ok else msg_bad))

counts = inventory["type"].value_counts()
for rec_type in ("A", "B"):
    n = int(counts.get(rec_type, 0))
    flag(m0["min_records_per_type"] <= n <= m0["max_records_per_type"],
         f"{n} records of type {rec_type}",
         f"{n} records of type {rec_type} (expected {m0['min_records_per_type']}-{m0['max_records_per_type']})")
flag("?" not in counts.index, "all file names follow A_n / B_n", "some files are neither A_* nor B_*")

lo = m0["expected_duration_s"] - m0["duration_tolerance_s"]
hi = m0["expected_duration_s"] + m0["duration_tolerance_s"]
for name, row in inventory.iterrows():
    flag(lo <= row["duration_s"] <= hi,
         f"{name}: duration {row['duration_s']} s",
         f"{name}: duration {row['duration_s']} s (protocol asks for ~{m0['expected_duration_s']:.0f} s)")

flag(inventory[["width", "height"]].drop_duplicates().shape[0] == 1,
     "all videos share the same resolution",
     "videos have different resolutions (the ROI in config.yaml is in pixels!)")

for name, s in scans.items():
    dt = np.diff(s["timestamps_s"])
    flag(dt.max() <= 1.5 * np.median(dt),
         f"{name}: regular frame spacing",
         f"{name}: irregular frame spacing (median {1000*np.median(dt):.1f} ms, max {1000*dt.max():.1f} ms) -> use timestamps")

## 0.3 Global brightness vs time (stimulus preview)

The mean gray level of each frame shows when the light changed. This is only a **sanity check** of the recording:
camera auto-exposure also modifies the global brightness, so the stimulus time used in Module 3 should be confirmed on the
images. Expected: a step **up** in `A_n`, a step **down** in `B_n`.

In [ ]:
onsets = []
n = len(paths)
fig, axes = plt.subplots(n, 1, figsize=(9, 2.6 * n), squeeze=False, sharex=False)
for ax, p in zip(axes.ravel(), paths):
    s = scans[p.name]
    kind = vio.record_type(p)
    t, b = s["timestamps_s"], s["brightness"]
    ax.plot(t, b, lw=1.2)
    if kind in ("A", "B"):
        est = detect_stimulus_onset(t, b, kind, m0["brightness_smooth_frames"])
        ax.axvline(est["onset_s"], color="tab:red", ls="--", label=f"estimated light change: {est['onset_s']:.2f} s")
        onsets.append({"file": p.name, "type": kind, "onset_s": round(est["onset_s"], 2),
                       "step_gray_levels": round(est["step_gray_levels"], 1)})
        ax.legend(loc="best", fontsize=8)
    ax.set_title(p.name, fontsize=10)
    ax.set_ylabel("mean gray level")
axes.ravel()[-1].set_xlabel("time (s)")
fig.tight_layout()
save_fig(fig, "m0_brightness_vs_time", cfg)
plt.show()
pd.DataFrame(onsets).set_index("file")

**Check:** the sign of `step_gray_levels` must be positive for A and negative for B. If not, the record may be mislabeled,
the light was not switched when expected, or the exposure compensation hides the step.

## 0.4 Contact sheet

Five frames evenly spaced along each record: eye framing, focus, head movement and blinks can be checked at a glance.

In [ ]:
n_cols = 5
fig, axes = plt.subplots(len(paths), n_cols, figsize=(2.6 * n_cols, 3.0 * len(paths)), squeeze=False)
for row, p in enumerate(paths):
    dur = scans[p.name]["duration_s"]
    times = np.linspace(0.0, max(dur - 0.1, 0.0), n_cols)
    frames = vio.read_frames_at_times(p, times)
    for col, (img, t_real, _) in enumerate(frames):
        ax = axes[row, col]
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{p.name}  t={t_real:.1f}s", fontsize=8)
fig.tight_layout()
save_fig(fig, "m0_contact_sheet", cfg)
plt.show()

## 0.5 Acquisition notes (fill in by hand)

Document the conditions: they explain differences between trials in Module 3.

| Item | Value |
|---|---|
| Camera / phone model | |
| Distance eye-camera (cm) | |
| Light source (flash, torch ...) and distance | |
| Room illumination in the dark phase | |
| Subject position (chin support?) | |
| Observations (blinks, movements ...) | |

In [ ]:
inventory.to_csv(cfg["_output_dir"] / "module0_video_inventory.csv")
pd.DataFrame(onsets).to_csv(cfg["_output_dir"] / "module0_stimulus_preview.csv", index=False)
print("Saved tables in", cfg["_output_dir"])